In [2]:
import pandas as pd

products_df = pd.read_csv("products.csv")   # Make sure the file name matches exactly
products_df.head()

,CustomerId,Product_Type,Enrollment_Date,Status
0,15634602,Credit Card,2022-05-04,Active
1,15634602,Insurance,2023-05-01,Active
2,15634602,Mutual Fund,2023-12-21,Active
3,15647311,Insurance,2022-01-24,Active
4,15647311,Mutual Fund,2022-12-09,Active


In [8]:
import pandas as pd

# --- 1. Basic summary ---
summary = {
    "Shape": products_df.shape,
    "Columns": list(products_df.columns),
    "Missing Values": products_df.isnull().sum().to_dict(),
    "Duplicate Rows": int(products_df.duplicated().sum())
}

# --- 2. Category distributions ---
product_count = products_df.groupby('CustomerId')['Product_Type'].nunique().reset_index()

# Rename for clarity
product_count.columns = ['CustomerId', 'Distinct_Product_Count']

status_counts = products_df['Status'].value_counts().to_dict()

# --- 3. Logical checks ---
duplicate_products = products_df.groupby(['CustomerId', 'Product_Type']).size().reset_index(name='count')
duplicate_products = duplicate_products[duplicate_products['count'] > 1]

invalid_dates = products_df[~pd.to_datetime(products_df['Enrollment_Date'], errors='coerce').notna()]

inactive_by_customer = products_df.groupby('CustomerId')['Status'].apply(lambda x: all(x == "Inactive"))

print("📊 Summary:", summary)
print("\n🛒 Product Type Distribution:", product_counts)
print("\n⚙️ Status Distribution:", status_counts)
print("\n🚨 Duplicate Product Assignments:", len(duplicate_products))
print("🚨 Invalid Enrollment Dates:", len(invalid_dates))
print("🚨 Customers with All Inactive Products:", int(inactive_by_customer.sum()))


📊 Summary: {'Shape': (20068, 4), 'Columns': ['CustomerId', 'Product_Type', 'Enrollment_Date', 'Status'], 'Missing Values': {'CustomerId': 0, 'Product_Type': 0, 'Enrollment_Date': 0, 'Status': 0}, 'Duplicate Rows': 0}

🛒 Product Type Distribution:       CustomerId  Product_Type
0       15565701             2
1       15565706             3
2       15565714             1
3       15565779             2
4       15565796             2
...          ...           ...
9995    15815628             2
9996    15815645             1
9997    15815656             3
9998    15815660             2
9999    15815690             1

[10000 rows x 2 columns]

⚙️ Status Distribution: {'Active': 16080, 'Inactive': 3988}

🚨 Duplicate Product Assignments: 0
🚨 Invalid Enrollment Dates: 0
🚨 Customers with All Inactive Products: 822


In [9]:
customers = pd.read_csv("Customers_Cleaned.csv")
merged_df = customers.merge(product_count, on='CustomerId', how='left')

# Fill missing counts with 0 (customers without products)
merged_df['Distinct_Product_Count'].fillna(0, inplace=True)

inactive_customers = products_df.groupby('CustomerId')['Status'].apply(lambda x: all(x == 'Inactive')).reset_index()
inactive_customers.columns = ['CustomerId', 'All_Inactive']
merged = customers.merge(inactive_customers, on='CustomerId', how='left')
merged['All_Inactive'].fillna(False, inplace=True)
merged.groupby('All_Inactive')['Exited'].value_counts(normalize=True)


C:\Users\HP\AppData\Local\Temp\ipykernel_3760\3693832431.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Distinct_Product_Count'].fillna(0, inplace=True)
C:\Users\HP\AppData\Local\Temp\ipykernel_3760\3693832431.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For

All_Inactive  Exited
False         0         0.791779
              1         0.208221
True          0         0.794486
              1         0.205514
Name: proportion, dtype: float64